# Flask 模型运行状态单元测试（详细中文注释版）

## 目标
验证 `flaskapi/app.py` 里的模型预测接口在不同运行状态下是否正常工作。

## 为什么要 mock
- 不依赖真实数据库（避免环境差异导致失败）
- 不依赖外部天气 API（避免网络波动）
- 不依赖真实模型文件的版本一致性（只测接口流程与逻辑）

## 本 notebook 覆盖的场景
1. 模型未加载时，`/predict` 是否返回 500 与错误信息。
2. 模型可用时，`/predict` 是否返回 200，且确实调用了模型。
3. `/predict/by-input` 在 DB 与 forecast 被 mock 后是否走通并返回 200。


In [1]:
# ====== 依赖导入 ======
import unittest
from unittest.mock import patch
from importlib.util import spec_from_file_location, module_from_spec
from pathlib import Path
from datetime import datetime, timedelta
import numpy as np
import sys

# 作用：自动定位项目根目录，避免你必须在固定目录启动 notebook。
# 逻辑：从当前目录向上查找，直到找到 flaskapi/app.py。
def find_project_root():
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "flaskapi" / "app.py").exists():
            return cand
    raise FileNotFoundError(f"Cannot locate project root from cwd={p}")

# 作用：动态加载 flaskapi/app.py 模块，得到 app 与路由函数。
# 重点：把 flaskapi 目录加入 sys.path，确保 app.py 的本地 import 能成功。
def load_app_module():
    project_root = find_project_root()
    app_path = project_root / "flaskapi" / "app.py"
    flaskapi_dir = project_root / "flaskapi"

    for path in (str(flaskapi_dir), str(project_root)):
        if path not in sys.path:
            sys.path.insert(0, path)

    spec = spec_from_file_location("flask_app_module", app_path)
    mod = module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

# FakeModel：测试替身模型。
# 用它代替真实 joblib 模型，避免模型文件/版本问题影响测试。
class FakeModel:
    def __init__(self):
        # called: 用来验证 Flask 端是否真的调用了 model.predict。
        self.called = False
        # last_columns: 记录 Flask 传给模型的特征列顺序，
        # 用于验证是否严格按 MODEL_FEATURES 组装输入。
        self.last_columns = None

    def predict(self, X):
        self.called = True
        self.last_columns = list(X.columns)
        # 返回固定浮点预测值，便于断言与复现。
        return np.array([6.2] * len(X))


In [2]:
class TestFlaskModelStatus(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        # 只加载一次 Flask 模块，提升测试速度。
        cls.mod = load_app_module()
        # 使用 Flask 内置测试客户端，不需要真实启动服务端口。
        cls.client = cls.mod.app.test_client()

    def setUp(self):
        # 每个测试前重置模型状态，避免测试间互相污染。
        self.mod.MODEL = FakeModel()
        # 明确设定模型所需特征，确保测试输入可控。
        self.mod.MODEL_FEATURES = [
            "number", "capacity", "day", "hour", "minute",
            "temp", "pressure", "humidity", "lng", "lat",
            "bikes_same_slot_mean"
        ]

    def test_predict_model_not_loaded_returns_500(self):
        # 场景1：模型不可用（模拟服务启动后模型加载失败）。
        self.mod.MODEL = None
        resp = self.client.post("/predict", json={})

        # 断言1：接口应返回 500（服务端内部状态异常）。
        self.assertEqual(resp.status_code, 500)
        # 断言2：错误文案包含 Model not loaded，便于前端/调用方定位问题。
        self.assertIn("Model not loaded", resp.get_json()["error"])

    def test_predict_uses_model_and_returns_200(self):
        # 场景2：提供完整特征，验证 /predict 主流程可用。
        payload = {f: 1 for f in self.mod.MODEL_FEATURES}
        resp = self.client.post("/predict", json=payload)

        # 断言1：HTTP 状态应成功。
        self.assertEqual(resp.status_code, 200)
        # 断言2：模型 predict 必须被调用，证明不是短路返回。
        self.assertTrue(self.mod.MODEL.called)
        # 断言3：送入模型的列顺序与 MODEL_FEATURES 一致。
        self.assertEqual(self.mod.MODEL.last_columns, self.mod.MODEL_FEATURES)

    def test_predict_by_input_model_running_with_mocks(self):
        # 场景3：测试 /predict/by-input 的整合流程。
        # 这个接口会调用 DB 特征与 forecast，因此这里都用 mock。
        now = datetime.now()
        target_dt = (now + timedelta(hours=2)).strftime("%Y-%m-%d %H:%M:%S")

        # 模拟数据库返回的站点特征。
        fake_db = {
            "number": 32,
            "capacity": 30,
            "lng": -6.26,
            "lat": 53.35,
            "bikes_1d_mean": 10.0,
            "bikes_same_slot_mean": 9.0,
        }
        # 模拟天气预测返回（与目标时间接近的一条）。
        fake_forecast = [{
            "dt": int((now + timedelta(hours=2)).timestamp()),
            "forecast_time": (now + timedelta(hours=2)).strftime("%Y-%m-%d %H:%M:%S"),
            "temperature": 12.0,
            "pressure": 1012,
            "humidity": 80,
        }]

        # patch.object 的含义：仅在 with 代码块内替换函数行为。
        # 退出 with 后自动恢复原函数。
        with patch.object(self.mod, "get_prediction_db_features", return_value=fake_db), \
             patch.object(self.mod, "get_forecast", return_value=fake_forecast):
            resp = self.client.get(f"/predict/by-input?station_id=32&datetime={target_dt}")

        # 断言1：整合接口成功返回。
        self.assertEqual(resp.status_code, 200)
        # 断言2：模型确实参与了预测。
        self.assertTrue(self.mod.MODEL.called)

        body = resp.get_json()
        # 断言3：返回体标识天气来源为 forecast，
        # 证明接口路径命中了我们 mock 的天气预测数据。
        self.assertEqual(body["weather_source"]["mode"], "forecast")


In [3]:
# 运行测试套件
# verbosity=2: 输出每个测试名，便于课堂演示和定位失败用例。
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestFlaskModelStatus)
unittest.TextTestRunner(verbosity=2).run(suite)


test_predict_by_input_model_running_with_mocks (__main__.TestFlaskModelStatus.test_predict_by_input_model_running_with_mocks) ... ok
test_predict_model_not_loaded_returns_500 (__main__.TestFlaskModelStatus.test_predict_model_not_loaded_returns_500) ... ok
test_predict_uses_model_and_returns_200 (__main__.TestFlaskModelStatus.test_predict_uses_model_and_returns_200) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.719s

OK


<unittest.runner.TextTestResult run=3 errors=0 failures=0>